In [1]:
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory

In [26]:
modelos = ['presidencial','senador']
requisitos = ['placa','mecanismo','horas_trabalho']

valores_maximos ={
    requisitos[0]:1293,
    requisitos[1]:600,
    requisitos[2]:3000,
}
informacoes={
    modelos[0]:{
        requisitos[0]:2.78,
        requisitos[1]:1,
        requisitos[2]:5,
    },
    modelos[1]:{
        requisitos[0]:2.22,
        requisitos[1]:1,
        requisitos[2]:3,
    }
}
lucro = {
    modelos[0]:149,
    modelos[1]:135,
}

In [27]:
model=pyo.ConcreteModel()

model.modelos=pyo.Set(initialize=modelos)
model.requisitos=pyo.Set(initialize=requisitos)
model.x = pyo.Var(model.modelos, bounds=(0,None), domain=pyo.NonNegativeIntegers)

def obj(model):
    return sum(model.x[m]*lucro[m] for m in model.modelos)
model.objetivo = pyo.Objective(rule=obj, sense=pyo.maximize)

def requisitos(model, r):
    return sum(model.x[m]*informacoes[m][r] for m in model.modelos)<=valores_maximos[r]
model.req = pyo.Constraint(model.requisitos,rule=requisitos)

# def mnin(model,m):
#     return model.x[m] >= 20
# model.mnin = pyo.Constraint(model.modelos,rule=mnin)

In [28]:
model.pprint()

2 Set Declarations
    modelos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'presidencial', 'senador'}
    requisitos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'placa', 'mecanismo', 'horas_trabalho'}

1 Var Declarations
    x : Size=2, Index=modelos
        Key          : Lower : Value : Upper : Fixed : Stale : Domain
        presidencial :     0 :  None :  None : False :  True : NonNegativeIntegers
             senador :     0 :  None :  None : False :  True : NonNegativeIntegers

1 Objective Declarations
    objetivo : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : maximize : 149*x[presidencial] + 135*x[senador]

1 Constraint Declarations
    req : Size=3, Index=requisitos, Active=True
        Key            : Lower : Body                                   : Upper  

In [29]:
opt = SolverFactory('cplex')
res = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\DECIV\AppData\Local\Temp\tmpyi8wh6xc.cplex.log' open.
CPLEX> Problem 'C:\Users\DECIV\AppData\Local\Temp\tmpnvmts54m.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\DECIV\AppData\Local\Temp\tmpnvmts54m.pyomo.lp
Objective sense      : Maximize
Variables            :       2  [General Integer: 2]
Objective nonzeros   :       2
Linear constraints   :       3  [Less: 3]
  Nonzeros           :       6
  RHS nonzeros       :       3

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 135.0000       

In [30]:
for m in model.x:
    print(f"Quantidade do modelo: {m}: {pyo.value(model.x[m])}")

print(f"Lucro: {pyo.value(model.objetivo)}")

Quantidade do modelo: presidencial: 1.0
Quantidade do modelo: senador: 581.0
Lucro: 78584.0
